# Subject-specific paper replication and evaluation-boundary diagnostic

This is a **new standalone notebook** for the Thinking Out Loud processed EEG derivatives. It pools INNER SPEECH sessions **within each subject**, never across subjects. It creates its own reproducible folds and has no dependency on previous experiment exports or saved splits.

Source: Tasin et al., IEEE Access 14 (2026), 10811–10827, [DOI 10.1109/ACCESS.2025.3644494](https://doi.org/10.1109/ACCESS.2025.3644494). The final local PDF and your project were inspected. The full paper refers to supplementary feature Tables T1–T3, which were not available in the inspected project or located public material. **Neither provided feature mode is an exact 191-feature reproduction.**

The deliberate ordering experiment **PAPER_ORDER — feature selection before CV; potential leakage** uses full-subject labels for selection before random 10-fold classifier CV. This is a diagnostic implementation of an ambiguous ordering, **not a claim about the authors' actual code**. `NESTED_SAFE` refits selection within training folds and holds out every outer-test trial. All reported performance comes from OOF predictions; paper-order OOF predictions still inherit the selection leakage.

No score targets, previous outer results or PCA appearance are used to optimize this notebook. Default: 10-feature comparisons, both boundaries, all available subjects, current combined features, VMD off. Larger stages are opt-in. Existing notebooks and source data are not modified.


## 1. Dependencies and inspected source provenance

Required: NumPy, pandas, SciPy, scikit-learn, matplotlib, MNE and PyWavelets. These must already exist in your environment; this notebook never runs pip. LightGBM is optional and explicitly skipped when absent. VMD uses installed `vmdpy` or `sktime.libs.vmdpy`; otherwise it is explicitly skipped with its effective status saved. `mrmr-selection` is **not required**: a previously implemented quotient selector is embedded below.

The audited current extractor's math is embedded unchanged. Its feature names alone are reformatted to `channel__feature`. The earlier `paper-reproduce.ipynb` supplies a clearly documented 152-feature approximation. Its old sample-window assumptions are **not** reused; the current audited FIF loader supplies the action interval.


In [ ]:
"""Subject-specific paper-order diagnostic and training-only evaluation."""
from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction
from types import SimpleNamespace
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED
from itertools import combinations
from importlib.metadata import version, PackageNotFoundError
import hashlib
import importlib.util
import json
import platform
import re
import warnings
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.signal import welch
from scipy.integrate import trapezoid
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, recall_score
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import pywt
BANDS = ((1.,4.), (4.,8.), (8.,13.), (13.,30.), (30.,45.))
PAPER_LABEL = 'PAPER_ORDER — feature selection before CV; potential leakage'
SAFE_LABEL = 'NESTED_SAFE'
LABELS = [0, 1, 2, 3]
REFERENCES = {
    ('mrmr', 10, 'svm'): (.6653, .6709),
    ('anova', 10, 'lgbm'): (.4594, .4630),
    ('mrmr', 590, 'svm'): (.7631, None),
    ('mrmr', 590, 'lda'): (.7191, None),
    ('mrmr', 590, 'etc'): (.7593, None),
    ('mrmr', 590, 'rf'): (.7342, None),
    ('mrmr', 590, 'lr'): (.7789, None),
    ('mrmr', 590, 'lgbm'): (.7533, None),
    ('mrmr', 590, 'lr5'): (.8113, .8112),
}
_sklearn_mutual_info_classif = mutual_info_classif

ENGINE_ID = 'f1c2025bfe88cbb552c1ec9e342a8d14d0ce5970f26e5d035e930ca65902f27b'
SOURCE_INSPECTION = {'current_source': 'Thinking_Out_Loud_Within_Session_ML_Ensemble / audited current source', 'current_source_sha256': 'c168a2f9351deb2b19d6060d219a4233aacdfb688a933d83ecf4a5f941994511', 'current_wavelet_sha256': 'e7d884d0c48ea58ba5102a0bfe27497b84a8eae01e4c7b064376d1bbdc814b82', 'paper_inspired_source': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\paper-reproduce.ipynb', 'paper_inspired_source_sha256': '0622a815f14cbd30f173a45f07a4859174a2eba9807eddbc827d3adb96add733', 'paper_exact_191_available': False, 'paper_pdf': 'C:\\Users\\User\\Programming\\Learning\\python\\braincodecamp\\braincodecamp2026\\presentations\\papers\\Ensemble_Machine_Learning_Model_for_Inner_Speech_Recognition_A_Subject-Specific_Investigation.pdf', 'doi': '10.1109/ACCESS.2025.3644494', 'current_default_reason': 'The saved compare-wavelet-ml-mrmr-anova run uses combined features.', 'loader_adaptation': 'Validated derivatives loader; retain pooled EEG and session IDs; crop by FIF times.', 'no_existing_files_modified': True}
print("Environment:", environment_info() if "environment_info" in globals() else platform.python_version())
print("Exact 191-feature specification available:", SOURCE_INSPECTION["paper_exact_191_available"])


## 2. Edit this one configuration cell

For a quick first subject, set `subjects=[1]`; `None` discovers all subjects. Keep `outer_folds=10`, `seed=42` for the requested experiment. `feature_mode="current"` uses your saved wavelet run's `combined` representation (120/channel, 15,360 total). `paper_inspired` uses the existing 50 TD + 24 FD + 78 DWT = **152/channel**, 19,456 total. DWT db4/level5 is an explicit approximation; the exact paper wavelet and fractal-length formula are unresolved.

`mrmr_relevance="f_stat"` matches the F-stat/Pearson quotient style of your current numeric mRMR implementation and is the faster initial diagnostic. Change it to `"mi"` for the existing mutual-information/Pearson approximation. The paper's exact relevance estimator, redundancy implementation, correlation floor and other hyperparameters are not specified sufficiently; **neither variant is claimed to be author-exact**. Every candidate feature is considered; there is no ANOVA prefilter or full feature-by-feature correlation matrix.

Enable stages with `run_sweep`, `run_models590`, `run_ensemble590`. `full_sweep=True` replaces the sparse grid with 10 plus every integer 20–610 and is expensive. Sweeps reuse maximum-K ranking prefixes for each fit set. `run_inner_k_choice=True` selects K and SVM parameters using inner training CV only. `tune_svm=True` also tunes SVM parameters within each outer training set for fixed-K experiments. The default fixed parameters were chosen before testing, not claimed to match paper hyperparameters.

For CASE 3, enable `run_matched_within_session`. For CASE 4, enable `run_feature_mode_comparison`; both are extra **K10** comparisons on matched trials/protocols. Changing feature mode or model settings requires rerunning setup/preparation and downstream cells. Results are separated by configuration; cache IDs include source content, code, labels, fit rows and versions.


In [ ]:
CONFIG = {
    "data_root": None,  # Discover one derivatives root under /kaggle/input; or set Path explicitly.
    "output_root": Path("/kaggle/working/paper_replication_subject_specific"),
    "subjects": None,   # All discovered subjects. Use [1] for the first diagnostic.
    "sessions": None,   # Pool all available sessions per subject.
    "seed": 42,
    "outer_folds": 10,
    "inner_folds": 3,
    "boundaries": ["PAPER_ORDER", "NESTED_SAFE"],
    "action_window_mne": (0.5, 3.0),  # Audited FIF clock, endpoint exclusive.
    "feature_mode": "current",       # current / paper_inspired
    "current_features": {
        "feature_set": "combined",    # Reuse current extractor: combined / eight_window / bandpower
        "n_windows": 8, "wavelet": "db2", "wavelet_level": 3,
        "ar_order": 4, "feature_sfreq": 128., "wavelet_bandpass": [2., 45.],
    },
    "paper_inspired_features": {"wavelet": "db4", "wavelet_level": 5, "spectral_high": 100.},
    "vmd_mode": "off",              # off / manual / auto_approx
    "vmd_manual_channels": {},        # Example: {1: ["A1", "B2"]}; use actual channel names.
    "vmd_robust_z": 4.0,              # OUR approximation, not a published threshold.
    "vmd_parameters": {"alpha": 2000., "tau": 0., "dc": 0, "init": 1, "tol": 1e-7},
    "mrmr_relevance": "f_stat",      # f_stat / mi; exact estimator unspecified in paper.
    "mrmr_corr_floor": 1e-6,
    "k_values": [10, 20, 50, 100, 128, 200, 300, 400, 500, 590, 610],
    "full_sweep": False,
    "run_k10": True,
    "include_anova_svm": True,
    "run_sweep": True,
    "run_models590": True,
    "run_ensemble590": True,
    "run_inner_k_choice": True,
    "run_matched_within_session": True,
    "run_feature_mode_comparison": True,
    "models590": ["svm", "lda", "etc", "rf", "lr", "lgbm"],
    "tune_svm": False,
    "svm_inner_grid": {"C": [0.1, 1., 10.], "kernel": ["rbf"], "gamma": ["scale"]},
    "models": {  # Reproducible implementation assumptions, not exact reported hyperparameters.
        "svm": {"C": 1., "kernel": "rbf", "gamma": "scale", "cache_size": 512},
        "lda": {"solver": "lsqr", "shrinkage": "auto"},
        "etc": {"n_estimators": 300, "max_features": "sqrt"},
        "rf": {"n_estimators": 300, "max_features": "sqrt"},
        "lr": {"C": 1., "solver": "lbfgs", "max_iter": 5000},
        "lgbm": {"n_estimators": 300, "learning_rate": .05, "num_leaves": 15,
                 "deterministic": True, "force_col_wise": True},
    },
    "lr5_members": [  # Paper-inspired ensemble hyperparameters — exact configs not reported.
        {"C": .1, "penalty": "l2", "solver": "lbfgs"},
        {"C": 1., "penalty": "l2", "solver": "lbfgs"},
        {"C": 10., "penalty": "l2", "solver": "lbfgs"},
        {"C": 1., "penalty": "l1", "solver": "liblinear"},
        {"C": 1., "penalty": "elasticnet", "solver": "saga", "l1_ratio": .5},
    ],
    "lr5_max_iter": 5000,
    "n_jobs": 1,
    "blas_threads": 1,  # Avoid nested oversubscription; modest memory use.
    "resume": True,
    "make_pca": True,
    "pca_subject": None, "pca_outer_fold": 1, "pca_k": 128,
    "pca_boundary": "NESTED_SAFE", "pca_include_test": True,
    "make_confusion_plots": False,   # CSV matrices are always saved; many plots for 10 subjects.
    "class_names": {0: "Up", 1: "Down", 2: "Right", 3: "Left"},
    "diagnostic_near_reference": .08,  # Descriptive CASE 1 tolerance, not statistical significance.
    "diagnostic_low_range": (.20, .40),
    "diagnostic_substantial_gain": .10,
}


## 3. Embedded reused feature definitions

These functions are copied from the inspected reliable project code; there is no runtime import from a previous notebook. Current combined features apply their existing trial-local 2–45 Hz filter and resample to 128 Hz, with eight windows, Burg AR4 and db2 level3 DWT entropy/log variance plus filtered-power fusion. The raw processed FIFs are not rereferenced/notched/ICA-filtered again.

The paper-inspired approximation uses signal/envelope statistics, Welch PSD descriptors and db4 level5 coefficient summaries. It retains the existing documented definitions, including normalized **geometric** length rather than asserting equivalence to an unavailable fractal formula. Every original trial remains one row; wavelet windows never become independent CV samples.


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()

def array_digest(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256(str((value.shape, value.dtype.str)).encode())
    h.update(memoryview(value).cast('B'))
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def bandpower_features(X, sfreq):
    n = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or n < 16:
        raise ValueError('Insufficient sampling rate or epoch length for declared bands.')
    freqs, density = welch(X, fs=sfreq, window='hann', nperseg=n,
                          noverlap=n//2, detrend='constant', scaling='density', axis=-1)
    values = []
    for low, high in BANDS:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError('Insufficient frequency bins.')
        power = np.trapezoid(density[...,mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def burg_coefficients(data, order=4):
    x = np.asarray(data, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Burg requires finite [signals, samples].')
    if not isinstance(order, (int,np.integer)) or not 1 <= order < x.shape[1]-1:
        raise ValueError('AR order must be positive and smaller than samples minus one.')
    x = x-x.mean(axis=-1,keepdims=True)
    scale = np.sqrt(np.mean(x*x,axis=-1,keepdims=True))
    x = x/np.maximum(scale,1e-15)
    forward,backward = x[:,1:].copy(),x[:,:-1].copy()
    coefficients = np.empty((len(x),0),dtype=np.float64)
    for lag in range(order):
        denominator = np.sum(forward*forward+backward*backward,axis=-1)
        reflection = np.divide(-2*np.sum(forward*backward,axis=-1),denominator,
            out=np.zeros(len(x)),where=denominator>1e-15)
        reflection = np.clip(reflection,-1+1e-7,1-1e-7)
        coefficients = np.concatenate([
            coefficients+reflection[:,None]*coefficients[:,::-1],reflection[:,None]],axis=1)
        updated_forward = forward+reflection[:,None]*backward
        updated_backward = backward+reflection[:,None]*forward
        forward,backward = updated_forward[:,1:],updated_backward[:,:-1]
    return -coefficients

def wavelet_statistics(coefficients):
    values = np.asarray(coefficients,dtype=np.float64)
    energy = values*values
    total = energy.sum(axis=-1,keepdims=True)
    probability = np.divide(energy,total,out=np.zeros_like(energy),where=total>1e-24)
    entropy = -np.sum(probability*np.log(np.maximum(probability,1e-300)),axis=-1)
    entropy = entropy/np.log(values.shape[-1]) if values.shape[-1]>1 else np.zeros_like(entropy)
    variance = np.log(np.maximum(np.var(values,axis=-1,ddof=0),1e-12))
    return np.clip(entropy,0.,1.),variance

def extract_trial_features(X,sfreq,ch_names,config):
    feature_set = config.get('feature_set','bandpower')
    rows = []
    def add(channel,window,family,component,start,stop):
        rows.append(dict(feature_index=len(rows),channel_index=int(channel),channel=ch_names[channel],
            window=int(window),family=family,component=component,
            start_time_mne=float(config['action_window_mne'][0]+start),
            stop_time_mne=float(config['action_window_mne'][0]+stop),
            feature_name=f'w{window:02d}__{ch_names[channel]}__{family}__{component}'))
    if feature_set=='bandpower':
        F = bandpower_features(X,sfreq)
        for channel in range(len(ch_names)):
            for lo,hi in BANDS:
                add(channel,0,'welch_log10_power',f'{lo:g}-{hi:g}Hz',0,X.shape[-1]/sfreq)
        return F,pd.DataFrame(rows),dict(feature_set=feature_set,units='volts',bands=BANDS,
            features=int(F.shape[1]),one_row_per_original_trial=True)
    if feature_set not in ['eight_window','combined']:
        raise ValueError('FEATURE_SET must be bandpower, eight_window, or combined.')
    import pywt
    n_windows = config.get('n_windows',8)
    wavelet = config.get('wavelet','db2')
    level = config.get('wavelet_level',3)
    ar_order = config.get('ar_order',4)
    target_rate = config.get('feature_sfreq',128.)
    lo,hi = config.get('wavelet_bandpass',[2.,45.])
    if n_windows!=8:
        raise ValueError('This reference implementation uses exactly eight windows.')
    if not 0 < lo < hi < min(sfreq,target_rate)/2:
        raise ValueError('Wavelet bandpass must be below both sampling Nyquist frequencies.')
    ratio = Fraction(float(target_rate)/float(sfreq)).limit_denominator(10000)
    samples = (X.shape[-1]*ratio.numerator+ratio.denominator-1)//ratio.denominator
    minimum_window = samples//n_windows
    maximum = pywt.dwt_max_level(minimum_window,wavelet)
    if not isinstance(level,(int,np.integer)) or not 1<=level<=maximum:
        raise ValueError(f'{wavelet} level {level} exceeds maximum {maximum} for {minimum_window}-sample windows.')
    if not isinstance(ar_order,(int,np.integer)) or not 1<=ar_order<minimum_window-1:
        raise ValueError('AR_ORDER is invalid for the window length.')
    microvolts = np.asarray(X,dtype=np.float32)*1e6
    sos = signal.butter(4,(lo,hi),btype='bandpass',fs=sfreq,output='sos')
    broadband = signal.sosfiltfilt(sos,microvolts,axis=-1).astype(np.float32)
    broadband = signal.resample_poly(broadband,ratio.numerator,ratio.denominator,axis=-1)
    components = [f'A{level}']+[f'D{i}' for i in range(level,0,-1)]
    descriptors = [('ar',f'lag{i}') for i in range(1,ar_order+1)]
    descriptors += [('wavelet_entropy',c) for c in components]
    descriptors += [('wavelet_log_variance',c) for c in components]
    windows = np.array_split(broadband,n_windows,axis=-1)
    matrices=[]
    offset=0
    for window,values in enumerate(windows,1):
        signals = values.reshape(-1,values.shape[-1])
        ar = burg_coefficients(signals,ar_order)
        coefficients = pywt.wavedec(signals.astype(np.float64),wavelet,
                                    mode='symmetric',level=level,axis=-1)
        summaries = [wavelet_statistics(c) for c in coefficients]
        entropy = np.stack([s[0] for s in summaries],axis=-1)
        variance = np.stack([s[1] for s in summaries],axis=-1)
        descriptor_values = np.concatenate([ar,entropy,variance],axis=-1)
        matrices.append(descriptor_values.reshape(len(X),-1))
        for channel in range(len(ch_names)):
            for family,component in descriptors:
                add(channel,window,family,component,offset/target_rate,(offset+values.shape[-1])/target_rate)
        offset += values.shape[-1]
    F = np.concatenate(matrices,axis=1)
    if feature_set=='combined':
        # Reference fusion: six filtered bands, whole epoch + three contiguous temporal bins.
        fusion_bands = ((2.,4.),(4.,8.),(8.,13.),(13.,20.),(20.,30.),(30.,45.))
        filtered = np.stack([signal.sosfiltfilt(signal.butter(4,band,btype='bandpass',fs=sfreq,output='sos'),
                                              microvolts,axis=-1).astype(np.float32) for band in fusion_bands],axis=1)
        edges = np.linspace(0,X.shape[-1],4).astype(int)
        bounds = [(0,X.shape[-1])]+list(zip(edges[:-1],edges[1:]))
        power = []
        for temporal,(start,stop) in enumerate(bounds):
            power.append(np.log(np.maximum(np.mean(filtered[...,start:stop]**2,axis=-1),1e-12)).reshape(len(X),-1))
            for low,high in fusion_bands:
                for channel in range(len(ch_names)):
                    add(channel,temporal,'filtered_log_power',f'{low:g}-{high:g}Hz',start/sfreq,stop/sfreq)
        F = np.concatenate([F,*power],axis=1)
    F = F.astype(np.float32)
    schema = pd.DataFrame(rows)
    if F.shape!=(len(X),len(schema)) or not np.isfinite(F).all() or schema.feature_name.duplicated().any():
        raise ValueError('Invalid descriptor values or schema.')
    info = dict(feature_set=feature_set,n_windows=n_windows,wavelet=wavelet,wavelet_level=level,
        wavelet_components=components,wavelet_mode='symmetric',boundary_coefficients='retained',
        ar_order=ar_order,ar_method='univariate_burg_demeaned',bandpass=[lo,hi],input_units='microvolts',
        input_sfreq=sfreq,feature_sfreq=target_rate,window_lengths=[w.shape[-1] for w in windows],
        entropy='normalized_negative_sum_p_log_p',variance='natural_log_population_variance_floor_1e-12',
        features=F.shape[1],one_row_per_original_trial=True,paper_exact_reproduction=False)
    return F,schema,info


In [ ]:
def safe_ratio(a, b):
    return a / np.maximum(np.abs(b), 1e-12)

def statistics25(x):
    """25 defined temporal descriptors, vectorized over leading dimensions."""
    x = np.asarray(x, np.float64)
    d = np.diff(x, axis=-1)
    dd = np.diff(d, axis=-1)
    mean, std = x.mean(axis=-1), x.std(axis=-1)
    absolute = np.abs(x)
    rms = np.sqrt(np.mean(x*x, axis=-1))
    variance = std**2
    mobility = np.sqrt(safe_ratio(np.var(d, axis=-1), variance))
    complexity = safe_ratio(np.sqrt(safe_ratio(np.var(dd, axis=-1), np.var(d, axis=-1))), mobility)
    # Central moments are explicitly zero for constant signals.
    standardized = (x - mean[..., None]) / np.maximum(std[..., None], 1e-12)
    values = [mean, std, variance, np.min(x, axis=-1), np.max(x, axis=-1),
              np.ptp(x, axis=-1), np.median(x, axis=-1),
              np.percentile(x, 75, axis=-1)-np.percentile(x, 25, axis=-1),
              absolute.mean(axis=-1), rms, np.sum(x*x, axis=-1),
              np.mean(standardized**3, axis=-1),
              np.where(std > 1e-12, np.mean(standardized**4, axis=-1)-3, 0),
              np.mean(np.abs(x-mean[..., None]), axis=-1),
              np.median(np.abs(x-np.median(x, axis=-1)[..., None]), axis=-1),
              np.mean(np.diff(np.signbit(x), axis=-1), axis=-1),
              np.sum(np.abs(d), axis=-1), np.mean(np.abs(d), axis=-1),
              np.mean(d*d, axis=-1), mobility, complexity,
              safe_ratio(np.max(absolute, axis=-1), rms),
              safe_ratio(rms, absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), np.mean(np.sqrt(absolute), axis=-1)**2)]
    names = ["mean", "std", "variance", "min", "max", "range", "median", "iqr",
             "mean_absolute", "rms", "energy", "skewness", "excess_kurtosis",
             "mean_absolute_deviation", "median_absolute_deviation", "zero_crossing_rate",
             "waveform_length", "mean_abs_difference", "mean_squared_difference",
             "hjorth_mobility", "hjorth_complexity", "crest_factor", "shape_factor",
             "impulse_factor", "clearance_factor"]
    return np.stack(values, axis=-1), names

def frequency24(x, sfreq, high):
    freq, psd = signal.welch(x, fs=sfreq, nperseg=min(256, x.shape[-1]), axis=-1)
    keep = (freq >= 0.5) & (freq <= high)
    f, p = freq[keep], psd[..., keep]
    if len(f) < 3:
        raise ValueError("Too few frequency bins")
    prob = safe_ratio(p, p.sum(axis=-1)[..., None])
    center = (prob*f).sum(axis=-1)
    width = np.sqrt((prob*(f-center[..., None])**2).sum(axis=-1))
    cum = np.cumsum(prob, axis=-1)
    roll = lambda q: f[np.argmax(cum >= q, axis=-1)]
    entropy = -np.sum(prob*np.log(np.maximum(prob, 1e-300)), axis=-1)/np.log(len(f))
    flatness = safe_ratio(np.exp(np.log(np.maximum(p, 1e-300)).mean(axis=-1)), p.mean(axis=-1))
    peak = p.max(axis=-1)
    weighted_std = np.maximum(width, 1e-12)[..., None]
    centered_power = p-p.mean(axis=-1)[..., None]
    sd_power = np.maximum(p.std(axis=-1), 1e-12)[..., None]
    power_z = centered_power/sd_power
    values = [trapezoid(p, f, axis=-1), f[p.argmax(axis=-1)], center, width,
              roll(0.5), roll(0.85), roll(0.95), entropy, flatness,
              safe_ratio(peak, p.mean(axis=-1)),
              (prob*((f-center[..., None])/weighted_std)**3).sum(axis=-1),
              (prob*((f-center[..., None])/weighted_std)**4).sum(axis=-1),
              np.mean(power_z**3, axis=-1), np.mean(power_z**4, axis=-1)-3]
    names = ["total_power", "peak_frequency", "centroid", "bandwidth", "median_frequency",
             "rolloff85", "rolloff95", "entropy", "flatness", "crest",
             "frequency_skewness", "frequency_kurtosis", "psd_skewness", "psd_excess_kurtosis"]
    total = trapezoid(p, f, axis=-1)
    for name, lo, hi in [("delta", .5, 4), ("theta", 4, 8), ("alpha", 8, 13),
                         ("beta", 13, 30), ("gamma", 30, high)]:
        mask = (freq >= lo) & (freq < hi if hi < high else freq <= hi)
        power = trapezoid(psd[..., mask], freq[mask], axis=-1)
        values.extend([power, safe_ratio(power, total)])
        names.extend([f"{name}_power", f"{name}_relative_power"])
    return np.stack(values, axis=-1), names

def wavelet_features(x, wavelet, level):
    if pywt.dwt_max_level(x.shape[-1], pywt.Wavelet(wavelet).dec_len) < level:
        raise ValueError("Window is too short for the requested DWT level")
    coefficients = pywt.wavedec(x, wavelet, level=level, axis=-1, mode="symmetric")
    labels = [f"A{level}"] + [f"D{i}" for i in range(level, 0, -1)]
    values, names = [], []
    for band, c in zip(labels, coefficients):
        d = np.diff(c, axis=-1)
        std = c.std(axis=-1)
        z = (c-c.mean(axis=-1)[..., None])/np.maximum(std[..., None], 1e-12)
        probability = safe_ratio(c*c, np.sum(c*c, axis=-1)[..., None])
        # Geometric length of amplitude-normalized coefficients; definition explicit
        # because the paper's "fractal length" formula is unavailable.
        norm = safe_ratio(c-c.min(axis=-1)[..., None], np.ptp(c, axis=-1)[..., None])
        geometric_length = np.sum(np.sqrt(np.diff(norm, axis=-1)**2 + (1/max(c.shape[-1]-1, 1))**2), axis=-1)
        band_values = [c.mean(axis=-1), np.mean(np.abs(c), axis=-1), std, std**2,
                       np.sqrt(np.mean(c*c, axis=-1)), np.mean(c*c, axis=-1),
                       np.sum(c*c, axis=-1), np.sum(np.abs(d), axis=-1),
                       np.ptp(c, axis=-1), np.mean(z**3, axis=-1),
                       np.where(std > 1e-12, np.mean(z**4, axis=-1)-3, 0),
                       -np.sum(probability*np.log(np.maximum(probability, 1e-300)), axis=-1),
                       geometric_length]
        band_names = ["mean", "mean_absolute", "std", "variance", "rms", "mean_power",
                      "energy", "waveform_length", "range", "skewness", "excess_kurtosis",
                      "energy_entropy", "normalized_geometric_length"]
        values.extend(band_values)
        names.extend([f"{band}_{n}" for n in band_names])
    return np.stack(values, axis=-1), names

def documented_features(trial, cfg):
    """50 TD + 24 FD + 78 TFD = 152/channel; deliberately not padded to 191."""
    td, td_names = statistics25(trial)
    env, env_names = statistics25(np.abs(signal.hilbert(trial, axis=-1)))
    fd, fd_names = frequency24(trial, cfg.expected_sfreq, cfg.spectral_high)
    tf, tf_names = wavelet_features(trial, cfg.wavelet, cfg.wavelet_level)
    matrix = np.concatenate([td, env, fd, tf], axis=-1)
    names = ([f"TD_signal_{n}" for n in td_names] + [f"TD_envelope_{n}" for n in env_names]
             + [f"FD_{n}" for n in fd_names] + [f"TFD_{n}" for n in tf_names])
    if matrix.shape[-1] != 152 or len(set(names)) != 152:
        raise AssertionError("Feature schema changed")
    if not np.isfinite(matrix).all():
        raise ValueError("Non-finite features; do not silently replace numerical failures")
    return matrix, names


## 4. Processed EEG loader, trial-local VMD and feature cache

The audited loader checks event/FIF alignment. Dataset event columns include raw cue sample, direction (0–3), condition and block; condition `1` selects INNER SPEECH. Subject/session come from filenames and original retained trial rows from `epochs.selection`. Session IDs and unique trial IDs remain attached to every feature row.

The current dataset audit maps original protocol action time `[1.0,3.5)` to the derivative FIF epoch clock `[0.5,3.0)`. Cropping uses `epochs.times` with an exclusive endpoint and verifies coverage, channel order and rate. At 256 Hz this is 640 samples. Printed summaries let you confirm your actual data; no sample index is guessed.

VMD defaults off. Manual channels apply to each trial of that subject. `auto_approx` computes robust deviations of channel means and log-standard deviations **within each trial**, without labels or a fitted cross-trial threshold. It is an approximation, and operates on the cropped action interval (paper scope is underspecified). Alpha/tau/init/tolerance are assumptions; six modes and lowest-frequency removal follow the described method. The cache records requested/effective VMD settings, backend and every application; skipping unavailable VMD is visible.


In [ ]:
def environment_info():
    result = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                  scipy=scipy.__version__, sklearn=sklearn.__version__)
    for name in ('mne', 'PyWavelets', 'lightgbm', 'vmdpy'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = 'not installed'
    return result

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.tmp')
    with temp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    temp.replace(path)

def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.csv.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def boundary_label(boundary):
    return PAPER_LABEL if boundary == 'PAPER_ORDER' else SAFE_LABEL

def discover_data_root(requested):
    if requested is not None:
        root = Path(requested)
        if not root.exists():
            raise FileNotFoundError(f'DATA_ROOT does not exist: {root}')
        return root
    base = Path('/kaggle/input')
    parents = {p.parent for p in base.rglob('sub-*_ses-*_eeg-epo.fif')} if base.exists() else set()
    if not parents:
        raise FileNotFoundError('No processed FIF files discovered. Set CONFIG["data_root"].')
    # Select the enclosing derivatives directory, never silently combine dataset copies.
    roots = {next((q for q in [p, *p.parents] if q.name == 'derivatives'), p) for p in parents}
    if len(roots) != 1:
        raise ValueError(f'Multiple dataset roots: {sorted(map(str, roots))}. Set data_root explicitly.')
    return roots.pop()

def discover_subjects(root):
    return sorted({int(re.match(r'sub-(\d+)_ses-', p.name)[1])
                   for p in Path(root).rglob('sub-*_ses-*_eeg-epo.fif')})

def vmd_backend():
    try:
        from vmdpy import VMD
        return VMD, 'vmdpy'
    except ImportError:
        try:
            from sktime.libs.vmdpy import VMD
            return VMD, 'sktime.libs.vmdpy'
        except ImportError:
            return None, 'unavailable'

def robust_z(x):
    med = np.median(x)
    return np.abs(x-med) / max(1.4826*np.median(np.abs(x-med)), 1e-12)

def apply_vmd(X, sfreq, names, subject, config):
    mode = config['vmd_mode']
    if mode == 'off':
        return X, dict(requested=mode, effective='off', backend='unused', applications=[])
    VMD, backend = vmd_backend()
    if VMD is None:
        print(f'Subject {subject:02d}: VMD requested but no installed backend; SKIPPED. No pip/internet used.', flush=True)
        return X, dict(requested=mode, effective='skipped_missing_backend', backend=backend, applications=[])
    cleaned = np.asarray(X, dtype=np.float64).copy()
    report = dict(requested=mode, effective=mode, backend=backend, applications=[],
                  approximation='Manual choices or robust per-trial channel mean/log-std detector; threshold and VMD parameters are assumptions.')
    manual = config['vmd_manual_channels'].get(subject, config['vmd_manual_channels'].get(str(subject), []))
    if any(name not in names for name in manual):
        raise ValueError(f'Unknown manual VMD channels: {set(manual)-set(names)}')
    for row, trial in enumerate(cleaned):
        if mode == 'manual':
            mask = np.array([name in manual for name in names])
        else:
            # Label-free, trial-local across-channel detector, no held-out-distribution fit.
            uv = trial*1e6
            mask = ((robust_z(uv.mean(axis=1)) > config['vmd_robust_z']) |
                    (robust_z(np.log(np.maximum(uv.std(axis=1), 1e-12))) > config['vmd_robust_z']))
        for col in np.flatnonzero(mask):
            x = trial[col]*1e6
            if np.std(x) < 1e-12:
                continue
            scale = max(float(np.sqrt(np.mean(x*x))), 1e-12)
            padded = np.pad(x/scale, (0, len(x)%2), mode='edge')
            p = config['vmd_parameters']
            modes, _, omega = VMD(padded, p['alpha'], p['tau'], 6, p['dc'], p['init'], p['tol'])
            if len(modes) != 6 or omega.ndim != 2:
                raise ValueError('Unexpected VMD output.')
            lowest = int(np.argmin(omega[-1]))
            clean = modes[np.arange(6) != lowest].sum(axis=0)[:len(x)] * scale / 1e6
            if not np.isfinite(clean).all():
                raise ValueError('Nonfinite VMD output.')
            trial[col] = clean
            report['applications'].append(dict(trial_row=row, channel=names[col], removed_mode=lowest,
                                               removed_hz=float(omega[-1, lowest]*sfreq)))
        if row == 0 or (row+1)%20 == 0:
            print(f'  VMD trial {row+1}/{len(X)}', flush=True)
    return cleaned.astype(np.float32), report

def feature_matrix(X, sfreq, names, mode, config):
    if mode == 'current':
        p = dict(config['current_features'], action_window_mne=config['action_window_mne'])
        F, schema, info = extract_trial_features(X, sfreq, names, p)
        schema = schema.copy()
        schema['original_feature_name'] = schema.feature_name
        schema['feature_name'] = schema.apply(lambda r: f'{r.channel}__w{r.window:02d}__{r.family}__{r.component}', axis=1)
        info['note'] = 'Current feature values and ordering reused exactly; names reformatted only.'
    else:
        cfg = SimpleNamespace(expected_sfreq=sfreq, **config['paper_inspired_features'])
        if cfg.wavelet_level != 5:
            raise ValueError('The existing documented_152 schema requires wavelet_level=5.')
        if not 30 < cfg.spectral_high <= sfreq/2:
            raise ValueError('spectral_high must be >30 and <= Nyquist.')
        # Existing approximation uses microvolts; explicit unit avoids scale-dependent numerical floors.
        values, feature_names = documented_features(X.astype(np.float64)*1e6, cfg)
        F = values.reshape(len(X), -1).astype(np.float32)
        schema = pd.DataFrame([dict(feature_index=i*len(feature_names)+j, channel_index=i,
                                   channel=channel, family=name.split('_')[0], component=name,
                                   feature_name=f'{channel}__{name}')
                              for i, channel in enumerate(names) for j, name in enumerate(feature_names)])
        info = dict(feature_mode=mode, features_per_channel=len(feature_names), units='microvolts',
                    wavelet=cfg.wavelet, wavelet_level=cfg.wavelet_level, spectral_high=cfg.spectral_high,
                    counts=dict(TD=50, FD=24, TFD=78), paper_exact_reproduction=False,
                    geometric_length='Amplitude-normalized polygonal length; not verified as the paper fractal-length formula.')
    if F.shape[1] != len(schema) or schema.feature_name.duplicated().any() or not np.isfinite(F).all():
        raise ValueError('Invalid feature schema or nonfinite values.')
    counts = schema.groupby('channel').size()
    if counts.nunique() != 1:
        raise ValueError('Feature counts differ across channels.')
    print(f'{"Paper-inspired feature set" if mode == "paper_inspired" else "Current feature set"}: '
          f'{int(counts.iloc[0])} features/channel; {F.shape[1]} total. Not an exact 191-feature replication.', flush=True)
    return F, schema, info

def prepare_subject_replication(subject, config):
    import mne
    root = Path(config['data_root'])
    files = sorted(root.rglob(f'sub-{subject:02d}_ses-*_eeg-epo.fif'))
    if config['sessions'] is not None:
        files = [p for p in files if int(re.search(r'_ses-(\d+)_', p.name)[1]) in config['sessions']]
    if not files:
        raise FileNotFoundError(f'No requested sessions for subject {subject}.')
    session_ids = [int(re.search(r'_ses-(\d+)_', p.name)[1]) for p in files]
    if len(set(session_ids)) != len(session_ids):
        raise ValueError('Duplicate subject/session FIF files; choose one dataset version.')
    if config['sessions'] is not None and set(session_ids) != set(config['sessions']):
        raise ValueError('Some requested sessions are missing.')
    paths = [q for p in files for q in (p, p.with_name(p.name.replace('_eeg-epo.fif','_events.dat')))]
    sources = [dict(path=str(p), sha256=file_sha256(p)) for p in paths]
    feature_config = {k: config[k] for k in ('action_window_mne', 'feature_mode', 'current_features',
                      'paper_inspired_features', 'vmd_mode', 'vmd_manual_channels', 'vmd_robust_z', 'vmd_parameters')}
    _, backend = vmd_backend() if config['vmd_mode'] != 'off' else (None, 'unused')
    key = digest(dict(subject=subject, sources=sources, feature_config=feature_config,
                      engine=ENGINE_ID, versions=environment_info(), vmd_backend=backend))
    cache = Path(config['output_root'])/'feature_cache'/key
    if config['resume'] and (cache/'manifest.json').exists():
        manifest = json.loads((cache/'manifest.json').read_text())
        F = np.load(cache/'features.npz')['F']
        metadata = pd.read_csv(cache/'metadata.csv')
        schema = pd.read_csv(cache/'schema.csv')
        if array_digest(F) != manifest['F_sha256'] or len(F) != len(metadata):
            raise ValueError('Feature cache integrity check failed.')
        print(f'Subject {subject:02d}: feature cache reused ({F.shape}); VMD: {manifest["vmd_effective"]}', flush=True)
        return dict(subject=subject, F=F, y=metadata.class_id.to_numpy(dtype=int), metadata=metadata,
                    schema=schema, manifest=manifest, feature_key=key)
    arrays, records, provenance, vmd_reports = [], [], [], []
    reference = None
    for path, session in zip(files, session_ids):
        event_path = path.with_name(path.name.replace('_eeg-epo.fif','_events.dat'))
        # Only the trusted dataset's event pickle is used, as in the audited existing loader.
        raw = np.asarray(pd.read_pickle(event_path))
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if raw.shape != (len(epochs),4) or not np.isfinite(raw).all() or not np.equal(raw,np.floor(raw)).all():
            raise ValueError(f'Invalid event dimensions or values: {event_path}')
        events = raw.astype(np.int64)
        if not np.array_equal(events[:,0],epochs.events[:,0]) or not np.array_equal(events[:,1]+31,epochs.events[:,2]):
            raise ValueError('FIF/event alignment failed.')
        if not np.isin(events[:,1],LABELS).all():
            raise ValueError('Unknown direction codes.')
        original_rows = np.asarray(epochs.selection)
        inner = np.flatnonzero(events[:,2] == 1)
        if not len(inner):
            raise ValueError('No INNER SPEECH trials in a requested session.')
        epochs = epochs[inner].load_data()
        epochs.pick(mne.pick_types(epochs.info,eeg=True,exclude=[]))
        names, sfreq = list(epochs.ch_names), float(epochs.info['sfreq'])
        if len(names) != 128:
            raise ValueError('This replication keeps all 128 EEG channels.')
        start, stop = config['action_window_mne']
        if start < epochs.times[0] or stop > epochs.times[-1]+1/sfreq+1e-9:
            raise ValueError('Requested action interval is outside the FIF epoch.')
        mask = (epochs.times >= start) & (epochs.times < stop)
        times = epochs.times[mask]
        if not len(times):
            raise ValueError('Empty action interval.')
        if reference is not None and (names != reference[0] or sfreq != reference[1] or not np.array_equal(times,reference[2])):
            raise ValueError('Sessions have different channel orders, rates or action times.')
        reference = names,sfreq,times
        X = epochs.get_data(copy=False)[:,:,mask].astype(np.float32)
        if not np.isfinite(X).all():
            raise ValueError('Nonfinite EEG.')
        X, report = apply_vmd(X,sfreq,names,subject,config)
        vmd_reports.append(dict(session=session, **report))
        arrays.append(X)
        for row, event in zip(original_rows[inner], events[inner]):
            records.append(dict(subject=subject,session=session,trial_index=int(row),cue_sample_raw=int(event[0]),
                                class_id=int(event[1]),condition='INNER',
                                trial_id=f'sub-{subject:02d}_ses-{session:02d}_row-{row:04d}'))
        provenance.append(dict(file=str(path),session=session,shape=list(X.shape),sfreq=sfreq,
                               first_time=float(times[0]),last_time=float(times[-1]),X_sha256=array_digest(X),
                               highpass=float(epochs.info['highpass']),lowpass=float(epochs.info['lowpass'])))
        print(f'Subject {subject:02d} session {session:02d}: X={X.shape}, sfreq={sfreq:g} Hz, INNER only.', flush=True)
    X = np.concatenate(arrays)
    metadata = pd.DataFrame(records)
    if metadata.trial_id.duplicated().any() or metadata.subject.nunique() != 1:
        raise ValueError('Trial IDs or subject separation failed.')
    F,schema,feature_info = feature_matrix(X,sfreq,names,config['feature_mode'],config)
    manifest = dict(subject=subject,pooled_shape=list(X.shape),sfreq=sfreq,channels=names,sources=sources,
                    provenance=provenance,feature_info=feature_info,feature_config=feature_config,
                    F_sha256=array_digest(F),vmd=vmd_reports,
                    vmd_effective=sorted({r['effective'] for r in vmd_reports}),feature_key=key,
                    action_interval_note='FIF epoch clock, audited cue-relative [0.5,3.0); original protocol action [1.0,3.5).')
    atomic_npz(cache/'features.npz',F=F)
    atomic_csv(metadata,cache/'metadata.csv')
    atomic_csv(schema,cache/'schema.csv')
    save_json(cache/'manifest.json',manifest)
    return dict(subject=subject,F=F,y=metadata.class_id.to_numpy(dtype=int),metadata=metadata,
                schema=schema,manifest=manifest,feature_key=key)

def dataset_summary(datasets, output_dir):
    rows = []
    for data in datasets.values():
        for session, meta in data['metadata'].groupby('session'):
            counts = meta.class_id.value_counts()
            rows.append(dict(subject=data['subject'],session=int(session),trials=len(meta),
                             trials_in_subject=len(data['y']),EEG_shape=str([len(meta),128,data['manifest']['pooled_shape'][2]]),
                             sfreq=data['manifest']['sfreq'],channels=128,
                             **{f'class_{i}':int(counts.get(i,0)) for i in LABELS}))
    table = pd.DataFrame(rows)
    atomic_csv(table,Path(output_dir)/'dataset_summary.csv')
    print('Subjects:', sorted(datasets), '\nSessions per subject:', table.groupby('subject').session.apply(list).to_dict())
    return table


## 5. Selection and classifier definitions

The embedded MRMR quotient considers all nonconstant candidate columns. Relevance is F-stat or sklearn mutual information; redundancy is mean absolute Pearson correlation against already chosen columns. Correlations are calculated incrementally, not stored in a large feature-by-feature matrix. MI uses deterministic column batches with progress messages (batching is an implementation detail, not an author specification).

Nonconstant columns require training standard deviation > `1e-12` in the declared feature units. No positive variance cutoff or ANOVA prefilter is applied. K is never silently reduced. Feature ranking is cached against its exact fit rows/labels; greedy prefixes are reused only at that same boundary.

Every model scaler is fitted to outer training only, even in PAPER_ORDER, so the main boundary comparison deliberately changes supervised selection. The five-LR ensemble uses exactly five configurable stratified training bootstraps, one scaler per bootstrap, explicit one-vs-rest LR and mean probabilities. This is soft voting, not a learned stacking meta-classifier. All five member settings are approximations.


In [ ]:
class MRMRQuotient(TransformerMixin, BaseEstimator):
    """Paper Eq.1 form: relevance / mean absolute correlation to selected features.

    MI and absolute Pearson correlation are explicit implementation choices.
    All candidates are considered; no relevance prefilter and no p-by-p matrix.
    """
    def __init__(self, k=590, relevance="mi", corr_floor=1e-6, seed=42):
        self.k, self.relevance, self.corr_floor, self.seed = k, relevance, corr_floor, seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        if len(X) != len(y) or not np.isfinite(X).all():
            raise ValueError("Invalid MRMR training input")
        self.n_features_in_ = X.shape[1]
        sd = X.std(axis=0)
        available_columns = np.flatnonzero(sd > 1e-12)
        if not len(available_columns):
            raise ValueError("All feature columns are constant")
        Z = (X[:, available_columns] - X[:, available_columns].mean(axis=0)) / sd[available_columns]
        if self.relevance == "mi":
            rel = mutual_info_classif(Z, y, random_state=self.seed)
        elif self.relevance == "f_stat":
            rel = f_classif(Z, y)[0]
        else:
            raise ValueError("Unknown MRMR relevance estimator")
        rel = np.nan_to_num(rel, nan=0, posinf=1e12, neginf=0)
        remaining = np.ones(len(rel), dtype=bool)
        redundancy_sum = np.zeros(len(rel), dtype=np.float64)
        chosen = []
        for step in range(min(int(self.k), len(rel))):
            score = rel.copy() if step == 0 else rel / np.maximum(redundancy_sum / step, self.corr_floor)
            score[~remaining] = -np.inf
            nxt = int(np.argmax(score))
            chosen.append(nxt)
            remaining[nxt] = False
            redundancy_sum += np.abs(Z.T @ Z[:, nxt] / len(Z))
        self.selected_indices_ = available_columns[np.asarray(chosen)]
        self.relevance_values_ = rel
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]


In [ ]:
def mutual_info_classif(X, y, random_state=42):
    """Same sklearn estimator, independent columns batched to expose progress."""
    parts = []
    for start in range(0, X.shape[1], 2048):
        stop = min(start+2048, X.shape[1])
        print(f'    MI relevance: features {start+1}–{stop}/{X.shape[1]}', flush=True)
        # Keep one RNG stream, as in a single call to sklearn.
        if start == 0:
            rng = np.random.RandomState(random_state)
        parts.append(_sklearn_mutual_info_classif(X[:,start:stop],y,random_state=rng))
    return np.concatenate(parts)

def cv_plan(data, config, scope='pooled_subject'):
    n = len(data['y'])
    units = [np.arange(n)] if scope == 'pooled_subject' else [g.index.to_numpy() for _,g in data['metadata'].groupby('session')]
    plan, visits = [], np.zeros(n,dtype=int)
    for unit in units:
        y = data['y'][unit]
        counts = np.bincount(y,minlength=4)
        if counts.min() < config['outer_folds']:
            raise ValueError(f'All four classes require >= {config["outer_folds"]} trials for this scope; counts={counts}. No silent fold reduction.')
        cv = StratifiedKFold(config['outer_folds'],shuffle=True,random_state=config['seed'])
        session = 0 if scope == 'pooled_subject' else int(data['metadata'].iloc[unit[0]].session)
        for fold,(a,b) in enumerate(cv.split(np.zeros(len(unit)),y),1):
            train,test = unit[a],unit[b]
            assert not np.intersect1d(train,test).size
            assert data['metadata'].iloc[train].subject.nunique() == 1
            if scope != 'pooled_subject':
                assert data['metadata'].iloc[train].session.nunique() == 1
            visits[test] += 1
            plan.append(dict(fold=fold,unit_session=session,train=train.tolist(),test=test.tolist(),selection_unit=unit.tolist()))
    assert np.all(visits==1), 'Exactly one held-out prediction per trial is required.'
    return plan

def ranking(data, fit_rows, selector, k_max, config, root):
    fit_rows = np.asarray(fit_rows,dtype=int)
    y = data['y'][fit_rows]
    key = digest(dict(feature_key=data['feature_key'],F=data['manifest']['F_sha256'],rows=fit_rows.tolist(),
                      y=array_digest(y),selector=selector,relevance=config['mrmr_relevance'],
                      corr_floor=config['mrmr_corr_floor'],seed=config['seed'],engine=ENGINE_ID,
                      versions=environment_info()))
    directory = Path(root)/'rank_cache'/key
    if config['resume'] and (directory/'ranking.npz').exists():
        with np.load(directory/'ranking.npz') as cached:
            indices = cached['indices']
        if len(indices) >= k_max:
            return indices[:k_max]
    F = data['F'][fit_rows].astype(np.float64)
    # Reused selector normalization is fitted at precisely fit_rows.
    eligible = np.flatnonzero(F.std(axis=0)>1e-12)
    if k_max > len(eligible):
        raise ValueError(f'K={k_max} exceeds {len(eligible)} eligible features; no silent clipping.')
    began = perf_counter()
    print(f'  Ranking {selector.upper()}: {len(fit_rows)} fit trials, {len(eligible)} candidates, K={k_max}; '
          f'relevance={config["mrmr_relevance"] if selector=="mrmr" else "f_stat"}', flush=True)
    with threadpool_limits(limits=config['blas_threads']):
        if selector == 'anova':
            values = np.nan_to_num(f_classif(F[:,eligible],y)[0],nan=0,posinf=1e12,neginf=0)
            indices = eligible[np.argsort(-values,kind='stable')[:k_max]]
        elif selector == 'mrmr':
            ranker = MRMRQuotient(k_max,config['mrmr_relevance'],config['mrmr_corr_floor'],config['seed'])
            ranker.fit(F,y)
            indices = ranker.selected_indices_
        else:
            raise ValueError('Unknown selector.')
    if len(indices)!=k_max or len(set(indices))!=k_max:
        raise ValueError('Selector did not return the requested unique feature count.')
    atomic_npz(directory/'ranking.npz',indices=indices)
    save_json(directory/'audit.json',dict(fit_rows=fit_rows.tolist(),fit_trial_ids=data['metadata'].iloc[fit_rows].trial_id.tolist(),
                                       labels_sha256=array_digest(y),selector=selector,k_max=k_max,eligible_features=len(eligible)))
    print(f'  Ranking done in {(perf_counter()-began)/60:.2f} minutes; prefixes reused for smaller K.', flush=True)
    return indices

class StratifiedLR5(ClassifierMixin, BaseEstimator):
    """Five configurable LR pipelines, stratified training-only bootstrap, mean probability vote."""
    def __init__(self, members=None, seed=42, max_iter=5000):
        self.members,self.seed,self.max_iter = members,seed,max_iter

    def fit(self, X, y):
        if self.members is None or len(self.members)!=5:
            raise ValueError('Exactly five LR member configurations are required.')
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        self.models_,self.bootstrap_indices_ = [],[]
        rng = np.random.default_rng(self.seed)
        for i, spec in enumerate(self.members):
            indices = np.concatenate([rng.choice(np.flatnonzero(y==c),np.sum(y==c),replace=True) for c in self.classes_])
            rng.shuffle(indices)
            params = dict(spec,max_iter=self.max_iter,random_state=self.seed+i)
            # Explicit OVR supports liblinear in multiclass tasks across sklearn versions.
            model = Pipeline([('scale',StandardScaler()),('lr',OneVsRestClassifier(LogisticRegression(**params),n_jobs=1))])
            model.fit(X[indices],y[indices])
            assert np.array_equal(model.classes_,self.classes_)
            self.models_.append(model)
            self.bootstrap_indices_.append(indices)
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_],axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X),axis=1)]

def make_model(name, config, override=None):
    seed = config['seed']
    params = dict(config['models'].get(name,{}))
    params.update(override or {})
    if name == 'svm':
        estimator,scale = SVC(**params),True
    elif name == 'lda':
        estimator,scale = LinearDiscriminantAnalysis(**params),True
    elif name == 'etc':
        estimator,scale = ExtraTreesClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'rf':
        estimator,scale = RandomForestClassifier(**params,random_state=seed,n_jobs=config['n_jobs']),False
    elif name == 'lr':
        estimator,scale = LogisticRegression(**params,random_state=seed),True
    elif name == 'lgbm':
        try:
            from lightgbm import LGBMClassifier
        except ImportError as exc:
            raise ImportError('LightGBM is unavailable; no substitute result is labeled LightGBM.') from exc
        estimator,scale = LGBMClassifier(**params,random_state=seed,n_jobs=config['n_jobs'],verbosity=-1),False
    elif name == 'lr5':
        return StratifiedLR5(config['lr5_members'],seed,config['lr5_max_iter'])
    else:
        raise ValueError(f'Unknown classifier: {name}')
    return Pipeline([('scale',StandardScaler() if scale else 'passthrough'),('model',estimator)])

def metrics(y, pred):
    rec = recall_score(y,pred,labels=LABELS,average=None,zero_division=0)
    return dict(accuracy=float(accuracy_score(y,pred)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),
                macro_f1=float(f1_score(y,pred,labels=LABELS,average='macro',zero_division=0)),
                **{f'recall_{i}':float(rec[i]) for i in LABELS})

def choose_inner(data, train, boundary, fixed_columns, selector, counts, config, root):
    counts = sorted(set(map(int,counts)))
    n_inner = config['inner_folds']
    if np.bincount(data['y'][train],minlength=4).min()<n_inner:
        raise ValueError('Insufficient outer-training examples for configured inner CV.')
    candidates = list(ParameterGrid(config['svm_inner_grid']))
    scores = np.zeros((len(counts),len(candidates),n_inner))
    cv = StratifiedKFold(n_inner,shuffle=True,random_state=config['seed'])
    audits = []
    for split,(a,b) in enumerate(cv.split(np.zeros(len(train)),data['y'][train])):
        inner_train,inner_valid = train[a],train[b]
        if boundary == 'NESTED_SAFE':
            columns = ranking(data,inner_train,selector,max(counts),config,root)
            selection_rows = inner_train
        else:
            columns = fixed_columns
            selection_rows = None # Pre-CV selection intentionally exposed these labels.
        assert not np.intersect1d(inner_train,inner_valid).size
        audits.append(dict(inner_fold=split+1,train=inner_train.tolist(),validation=inner_valid.tolist(),
                           selection_rows=None if selection_rows is None else selection_rows.tolist()))
        for ki,k in enumerate(counts):
            for pi,params in enumerate(candidates):
                model = make_model('svm',config,params)
                with threadpool_limits(limits=config['blas_threads']):
                    model.fit(data['F'][inner_train][:,columns[:k]],data['y'][inner_train])
                    p = model.predict(data['F'][inner_valid][:,columns[:k]])
                scores[ki,pi,split] = balanced_accuracy_score(data['y'][inner_valid],p)
    ki,pi = np.unravel_index(np.argmax(scores.mean(axis=2)),scores.shape[:2])
    return counts[ki],candidates[pi],dict(inner_splits=audits,candidates=candidates,k_values=counts,
                                       mean_balanced_accuracy=scores.mean(axis=2).tolist(),selected_from='inner validation only')


## 6. Evaluation engine and checkpoint integrity

The same shuffled stratified outer plan is regenerated deterministically for both boundaries. Separate subject-specific models are fit in every fold. Fixed K and fixed model settings need no inner search; the label NESTED_SAFE identifies the training-only boundary. When tuning is enabled, every inner split refits selection and scaling on its own training rows. K selection is available only through the separate safe inner-selection stage.

PAPER_ORDER intentionally selects with full-subject labels; its inner tuning, if enabled, still inherits this exposure and remains labeled potentially leaky. Fold checkpoints save fit/test rows, chosen supports, training metrics and bootstrap global indices. Resume identities include data/code/parameters/splits/versions. Partial checkpoints never become final subject results until exactly one prediction exists per trial.


In [ ]:
def evaluate(data, spec, boundary, config, root, scope='pooled_subject'):
    selector, requested_k, classifier = spec['selector'],spec['k'],spec['classifier']
    plan = cv_plan(data,config,scope)
    setting = dict(subject=data['subject'],feature_mode=config['feature_mode'],scope=scope,boundary=boundary,
                   selection_boundary=boundary_label(boundary),**spec)
    result_key = digest(dict(setting=setting,feature_key=data['feature_key'],plan=plan,
                             evaluation_config={k:config[k] for k in ('seed','outer_folds','inner_folds','mrmr_relevance',
                             'mrmr_corr_floor','models','lr5_members','lr5_max_iter','tune_svm','svm_inner_grid','k_values')},
                             engine=ENGINE_ID,versions=environment_info()))
    directory = Path(root)/'runs'/result_key
    directory.mkdir(parents=True,exist_ok=True)
    k_max = max(config['k_values']) if requested_k == 'inner' else int(requested_k)
    n = len(data['y'])
    predictions = np.full(n,-1,dtype=int)
    probabilities = np.full((n,4),np.nan)
    outer_fold,unit_session,visits = np.zeros(n,dtype=int),np.zeros(n,dtype=int),np.zeros(n,dtype=int)
    selections,fold_rows,audits = [],[],[]
    save_json(directory/'setting.json',dict(**setting,result_id=result_key,feature_key=data['feature_key']))
    save_json(directory/'plan.json',plan)
    for fold in plan:
        train,test = np.asarray(fold['train']),np.asarray(fold['test'])
        selection_rows = np.asarray(fold['selection_unit']) if boundary=='PAPER_ORDER' else train
        token = f"ses-{fold['unit_session']:02d}_fold-{fold['fold']:02d}"
        checkpoint = directory/f'{token}.npz'
        audit_path = directory/f'{token}.json'
        if config['resume'] and checkpoint.exists() and audit_path.exists():
            with np.load(checkpoint) as saved:
                pred,cols,prob = saved['pred'],saved['columns'],saved['probabilities']
            audit = json.loads(audit_path.read_text())
            if audit['test_rows'] != test.tolist() or audit['train_rows'] != train.tolist() or len(pred)!=len(test):
                raise ValueError('Checkpoint split mismatch.')
            chosen_k = audit['chosen_k']
            print(f"  Subject {data['subject']:02d}, fold {fold['fold']:02d}: resumed {boundary_label(boundary)}",flush=True)
        else:
            start = perf_counter()
            print(f"Subject {data['subject']:02d}, {scope}, {selector.upper()}, K={requested_k}, {classifier.upper()}, "
                  f"{boundary_label(boundary)}, fold {fold['fold']}/{config['outer_folds']} (session unit {fold['unit_session']}).",flush=True)
            cols = ranking(data,selection_rows,selector,k_max,config,root)
            chosen_k,params,inner_audit = k_max,{},None
            if classifier=='svm' and (config['tune_svm'] or requested_k=='inner'):
                counts = config['k_values'] if requested_k=='inner' else [k_max]
                chosen_k,params,inner_audit = choose_inner(data,train,boundary,cols,selector,counts,config,root)
            cols = cols[:chosen_k]
            if boundary=='NESTED_SAFE':
                assert not np.intersect1d(selection_rows,test).size
            model = make_model(classifier,config,params)
            with threadpool_limits(limits=config['blas_threads']):
                model.fit(data['F'][train][:,cols],data['y'][train])
                pred = model.predict(data['F'][test][:,cols])
                train_pred = model.predict(data['F'][train][:,cols])
                prob = model.predict_proba(data['F'][test][:,cols]) if hasattr(model,'predict_proba') else np.empty((len(test),0))
            bootstrap = []
            if classifier=='lr5':
                bootstrap = [train[x].tolist() for x in model.bootstrap_indices_]
                assert all(set(x).issubset(set(train)) and not set(x).intersection(test) for x in bootstrap)
            audit = dict(train_rows=train.tolist(),test_rows=test.tolist(),selection_fit_rows=selection_rows.tolist(),
                         scaler_fit_rows=train.tolist(),bootstrap_global_rows=bootstrap,
                         bootstrap_scalers_fit_on='each bootstrap sample' if classifier=='lr5' else 'outer train',
                         chosen_k=int(chosen_k),model_overrides=params,inner_selection=inner_audit,
                         train_scores=metrics(data['y'][train],train_pred),elapsed_seconds=perf_counter()-start,
                         selection_boundary=boundary_label(boundary))
            atomic_npz(checkpoint,pred=pred,columns=cols,probabilities=prob)
            save_json(audit_path,audit)
            print(f"  Held-out BA={balanced_accuracy_score(data['y'][test],pred):.3f}; fold finished in {audit['elapsed_seconds']/60:.2f} minutes.",flush=True)
        predictions[test] = pred
        if prob.shape == (len(test),4):
            probabilities[test] = prob
        outer_fold[test] = fold['fold']
        unit_session[test] = fold['unit_session']
        visits[test] += 1
        if boundary=='NESTED_SAFE':
            assert not set(audit['selection_fit_rows']).intersection(test)
        assert set(audit['scaler_fit_rows']).issubset(train)
        assert all(set(x).issubset(train) for x in audit['bootstrap_global_rows'])
        fold_rows.append(dict(**setting,fold=fold['fold'],unit_session=fold['unit_session'],chosen_k=chosen_k,
                              train_accuracy=audit['train_scores']['accuracy'],train_balanced_accuracy=audit['train_scores']['balanced_accuracy'],
                              **metrics(data['y'][test],pred)))
        selections.extend([dict(result_id=result_key,fold=fold['fold'],unit_session=fold['unit_session'],rank=i+1,
                                feature_index=int(col),feature_name=data['schema'].iloc[col].feature_name,
                                channel=data['schema'].iloc[col].channel,**setting) for i,col in enumerate(cols)])
        audits.append(audit)
    assert np.all(visits==1) and np.isin(predictions,LABELS).all()
    assert data['metadata'].subject.nunique()==1
    oof = data['metadata'].copy()
    oof['y_true'],oof['y_pred'],oof['outer_fold'],oof['unit_session'] = data['y'],predictions,outer_fold,unit_session
    if np.isfinite(probabilities).all():
        for label in LABELS:
            oof[f'probability_{label}'] = probabilities[:,label]
    for name,value in setting.items():
        oof[name]=value
    row = dict(**setting,result_id=result_key,n_trials=n,OOF_predictions_per_trial=1,
               vmd_effective=str(data['manifest']['vmd_effective']),**metrics(data['y'],predictions))
    selection_frame = pd.DataFrame(selections)
    atomic_csv(oof,directory/'oof.csv')
    atomic_csv(selection_frame,directory/'selected_features.csv')
    atomic_csv(pd.DataFrame(fold_rows),directory/'fold_scores.csv')
    atomic_csv(pd.DataFrame(confusion_matrix(data['y'],predictions,labels=LABELS),columns=[f'pred_{i}' for i in LABELS])
               .assign(true_class=LABELS),directory/'confusion.csv')
    save_json(directory/'result.json',row)
    return dict(row=row,oof=oof,selections=selection_frame,fold_scores=pd.DataFrame(fold_rows),directory=directory,audits=audits)

def specs_for_stage(stage, config):
    if stage=='k10':
        result = [dict(stage=stage,selector='mrmr',k=10,classifier='svm'),
                  dict(stage=stage,selector='anova',k=10,classifier='lgbm')]
        if config['include_anova_svm']:
            result.append(dict(stage=stage,selector='anova',k=10,classifier='svm'))
        return result
    if stage=='sweep':
        return [dict(stage=stage,selector='mrmr',k=k,classifier='svm') for k in config['k_values']]
    if stage=='models590':
        return [dict(stage=stage,selector='mrmr',k=590,classifier=m) for m in config['models590']]
    if stage=='ensemble590':
        return [dict(stage=stage,selector='mrmr',k=590,classifier='lr5')]
    if stage=='inner_k_choice':
        return [dict(stage=stage,selector='mrmr',k='inner',classifier='svm')]
    raise ValueError('Unknown stage.')

def run_stage(datasets, stage, config, root, active, status, scope='pooled_subject'):
    specs = specs_for_stage(stage,config)
    for subject,data in datasets.items():
        if stage=='sweep':
            # Greedy MRMR prefixes are identical for a fixed fit set. Rank to maximum K ONCE.
            seen=set()
            for boundary in config['boundaries']:
                print(f'Precomputing sweep ranks: subject {subject:02d}, {scope}, {boundary_label(boundary)}',flush=True)
                for fold in cv_plan(data,config,scope):
                    rows=fold['selection_unit'] if boundary=='PAPER_ORDER' else fold['train']
                    identity=tuple(rows)
                    if identity not in seen:
                        ranking(data,rows,'mrmr',max(config['k_values']),config,root)
                        seen.add(identity)
        for spec in specs:
            if spec['classifier']=='lgbm' and importlib.util.find_spec('lightgbm') is None:
                record = dict(subject=subject,feature_mode=config['feature_mode'],scope=scope,**spec,
                              status='skipped',reason='LightGBM not installed; no replacement used.')
                status.append(record)
                print(record['reason'],flush=True)
                continue
            boundaries = ['NESTED_SAFE'] if stage=='inner_k_choice' else config['boundaries']
            for boundary in boundaries:
                result = evaluate(data,spec,boundary,config,root,scope)
                active[result['row']['result_id']] = result
                atomic_csv(pd.DataFrame([x['row'] for x in active.values()]),Path(root)/'active_subject_results.csv')
    atomic_csv(pd.DataFrame(status,columns=['subject','feature_mode','scope','stage','selector','k','classifier','status','reason']),Path(root)/'stage_status.csv')
    return pd.DataFrame([x['row'] for x in active.values() if x['row']['stage']==stage])

def aggregate_subjects(scores):
    if scores.empty:
        return scores.copy()
    keys = ['feature_mode','scope','stage','selector','k','classifier','selection_boundary']
    parts = []
    for setting,frame in scores.groupby(keys,dropna=False):
        if frame.subject.duplicated().any():
            raise ValueError('Duplicate subject observations for an aggregation cell.')
        row = dict(zip(keys,setting),n_subjects=frame.subject.nunique(),subjects=','.join(map(str,sorted(frame.subject))))
        for metric in ('accuracy','balanced_accuracy','macro_f1',*[f'recall_{i}' for i in LABELS]):
            row[f'{metric}_mean']=frame[metric].mean()
            row[f'{metric}_sd']=frame[metric].std(ddof=1)
        ref = REFERENCES.get((row['selector'],row['k'],row['classifier']),(None,None))
        row['paper_reference_accuracy'],row['paper_reference_f1']=ref
        row['paper_reference_ba']=None
        parts.append(row)
    return pd.DataFrame(parts)

def stability_tables(active, root):
    frequency,jaccard = [],[]
    for result in active.values():
        row = result['row']
        if row['boundary']!='NESTED_SAFE' or row['scope']!='pooled_subject':
            continue
        s = result['selections']
        folds = [set(frame.feature_index) for _,frame in s.groupby('fold')]
        common = {k:row[k] for k in ('result_id','subject','feature_mode','stage','selector','k','classifier','selection_boundary')}
        for (a,x),(b,y) in combinations(enumerate(folds,1),2):
            jaccard.append(dict(**common,fold_a=a,fold_b=b,jaccard=len(x&y)/len(x|y)))
        for feature,frame in s.groupby('feature_index'):
            frequency.append(dict(**common,feature_index=feature,feature=frame.feature_name.iloc[0],channel=frame.channel.iloc[0],
                                  selected_in_x_of_10_folds=frame.fold.nunique(),total_outer_folds=len(folds),
                                  selection_fraction=frame.fold.nunique()/len(folds)))
    frequency = pd.DataFrame(frequency)
    if not frequency.empty:
        frequency = frequency.sort_values(['subject','selection_fraction'],ascending=[True,False])
    jaccard = pd.DataFrame(jaccard)
    atomic_csv(frequency,Path(root)/'D_feature_stability.csv')
    atomic_csv(jaccard,Path(root)/'D_pairwise_jaccard.csv')
    return frequency,jaccard

def boundary_gaps(scores):
    if scores.empty:
        return pd.DataFrame()
    keys = ['subject','feature_mode','scope','stage','selector','k','classifier']
    a=scores[scores.boundary=='PAPER_ORDER']
    b=scores[scores.boundary=='NESTED_SAFE']
    joined = a.merge(b,on=keys,suffixes=('_paper_order','_nested_safe'),validate='one_to_one')
    cols = keys.copy()
    for metric in ('accuracy','balanced_accuracy','macro_f1'):
        joined[f'{metric}_gap']=joined[f'{metric}_paper_order']-joined[f'{metric}_nested_safe']
        cols.extend([f'{metric}_paper_order',f'{metric}_nested_safe',f'{metric}_gap'])
    joined['paper_order_label']=PAPER_LABEL
    return joined[cols+['paper_order_label']]

def final_interpretation(scores, config):
    lines = ['Exploratory diagnostics; these controlled comparisons do not establish how the paper code was implemented.']
    if scores.empty:
        return lines+['No completed experiments.']
    main = scores[(scores.stage=='k10')&(scores.selector=='mrmr')&(scores.classifier=='svm')&(scores.scope=='pooled_subject')]
    # Paired subjects only, same representation and actual trial IDs (guaranteed by active evaluations).
    for feature_mode,frame in main.groupby('feature_mode'):
        gap = boundary_gaps(frame)
        if gap.empty:
            lines.append(f'{feature_mode}: both modes are required for CASE 1/2.')
            continue
        po,sa = gap.accuracy_paper_order.mean(),gap.accuracy_nested_safe.mean()
        if abs(po-.6653)<=config['diagnostic_near_reference'] and config['diagnostic_low_range'][0]<=sa<=config['diagnostic_low_range'][1] and po-sa>=.15:
            lines.append(f'CASE 1 ({feature_mode}): PAPER_ORDER Acc={po:.1%}, NESTED_SAFE Acc={sa:.1%}. Selection before CV is a plausible major contributor in OUR implementation; it does not prove leakage in the published experiment.')
        elif all(config['diagnostic_low_range'][0]<=x<=config['diagnostic_low_range'][1] for x in (po,sa)):
            lines.append(f'CASE 2 ({feature_mode}): both accuracies are low ({po:.1%}/{sa:.1%}). Investigate exact T1–T3 feature definitions, VMD assumptions and representation; changing the evaluation order alone did not explain the gap.')
        else:
            lines.append(f'{feature_mode}: K10 MRMR/SVM Acc PAPER_ORDER={po:.1%}, NESTED_SAFE={sa:.1%}, paired gap={po-sa:+.1%}; neither CASE 1 nor CASE 2 threshold is met.')
    matches=['subject','feature_mode','stage','selector','k','classifier','boundary']
    pooled = scores[(scores.scope=='pooled_subject')&(scores.stage=='k10')]
    within = scores[(scores.scope=='within_session')&(scores.stage=='k10')]
    matched=pooled.merge(within,on=matches,suffixes=('_pooled','_within'))
    if matched.empty:
        lines.append('CASE 3 not assessed: enable run_matched_within_session for the same features/models/trials. Earlier 25–30% results with other extractors are not a matched baseline.')
    else:
        for (feature,selector,model),g in matched.groupby(['feature_mode','selector','classifier']):
            gains=g.assign(gain=g.balanced_accuracy_pooled-g.balanced_accuracy_within).groupby('boundary').gain.mean()
            if len(gains)==2 and (gains>=config['diagnostic_substantial_gain']).all():
                lines.append(f'CASE 3 ({feature}/{selector}/{model}): both modes improve BA after pooling ({gains.to_dict()}); increased training scope is a plausible contributor, not isolated session-stationarity evidence.')
            else:
                lines.append(f'Matched pooling BA differences ({feature}/{selector}/{model}): {gains.to_dict()}. CASE 3 threshold not met in both modes.')
    matches=['subject','scope','stage','selector','k','classifier','boundary']
    a=scores[(scores.feature_mode=='paper_inspired')&(scores.stage=='k10')&(scores.scope=='pooled_subject')]
    b=scores[(scores.feature_mode=='current')&(scores.stage=='k10')&(scores.scope=='pooled_subject')]
    match=a.merge(b,on=matches,suffixes=('_inspired','_current'))
    if match.empty:
        lines.append('CASE 4 not assessed: enable run_feature_mode_comparison to evaluate both representations on identical folds. The 152-feature approximation cannot test the missing exact 191 specification.')
    else:
        for (selector,model),g in match.groupby(['selector','classifier']):
            safe=g[g.boundary=='NESTED_SAFE']
            gain=(safe.balanced_accuracy_inspired-safe.balanced_accuracy_current).mean()
            if np.isfinite(gain) and gain>=config['diagnostic_substantial_gain']:
                lines.append(f'CASE 4 ({selector}/{model}): paper-inspired improves safe BA by {gain:.1%}; representation is a plausible contributor. Confirm with a frozen protocol and an independent evaluation.')
            else:
                lines.append(f'Matched representation safe BA difference ({selector}/{model}): {gain:+.1%}; CASE 4 threshold not met.')
    lines += ['Stability is descriptive: repeat selection can reflect correlated artifacts and does not prove physiology.',
              'Accuracy and balanced accuracy are different metrics. Paper F1 averaging is insufficiently specified; our F1 is explicitly macro.',
              'Sweeps/plots are exploratory. Do not choose settings from outer OOF scores and report them as an unbiased final estimate.']
    return lines

def export_tables(active, status, config, root):
    scores=pd.DataFrame([r['row'] for r in active.values()])
    A=aggregate_subjects(scores)
    B=scores
    C=scores[scores.stage=='sweep'].copy() if not scores.empty else pd.DataFrame()
    E=boundary_gaps(scores)
    for name,frame in [('A_paper_benchmark',A),('B_per_subject',B),('C_k_sweep',C),('E_boundary_gap',E)]:
        atomic_csv(frame,Path(root)/f'{name}.csv')
    D,J=stability_tables(active,root)
    session_rows=[]
    for result in active.values():
        row=result['row']
        for session,g in result['oof'].groupby('session'):
            session_rows.append(dict(**{k:row[k] for k in ('result_id','subject','feature_mode','scope','stage','selector','k','classifier','selection_boundary')},
                                     session=session,n_trials=len(g),**metrics(g.y_true,g.y_pred)))
    sessions=pd.DataFrame(session_rows)
    atomic_csv(sessions,Path(root)/'session_oof_scores.csv')
    atomic_csv(pd.DataFrame(status),Path(root)/'stage_status.csv')
    lines=final_interpretation(scores,config)
    (Path(root)/'diagnostic_interpretation.txt').write_text('\n\n'.join(lines),encoding='utf-8')
    save_json(Path(root)/'run_manifest.json',dict(config=config,versions=environment_info(),engine_id=ENGINE_ID,
               source_inspection=SOURCE_INSPECTION,active_result_ids=list(active),requested_subjects=config['subjects'],
               tables_use='Only active completed evaluations; folds are not independent subjects.',
               metric_units='CSV scores are proportions; paper refs are accuracy, not balanced accuracy.'))
    for line in lines:
        print(line+'\n')
    return dict(A=A,B=B,C=C,D=D,E=E,jaccard=J,sessions=sessions)


In [ ]:
def plot_pca(data, config, root):
    boundary=config['pca_boundary']
    plan=cv_plan(data,config)
    fold=plan[config['pca_outer_fold']-1]
    train,test=np.asarray(fold['train']),np.asarray(fold['test'])
    fit_rows=np.asarray(fold['selection_unit']) if boundary=='PAPER_ORDER' else train
    cols={sel:ranking(data,fit_rows,sel,config['pca_k'],config,root) for sel in ('anova','mrmr')}
    fig,axes=plt.subplots(1,3,figsize=(16,4.8))
    for ax,(name,columns) in zip(axes,[('Before selection',np.arange(data['F'].shape[1])),('ANOVA',cols['anova']),('MRMR',cols['mrmr'])]):
        scaler=StandardScaler().fit(data['F'][train][:,columns])
        a=scaler.transform(data['F'][train][:,columns])
        b=scaler.transform(data['F'][test][:,columns])
        pca=PCA(n_components=2,svd_solver='full').fit(a)
        a,b=pca.transform(a),pca.transform(b)
        for label in LABELS:
            color=f'C{label}'
            tr,te=data['y'][train]==label,data['y'][test]==label
            ax.scatter(a[tr,0],a[tr,1],c=color,s=27,label=config['class_names'][label],alpha=.8)
            if config['pca_include_test']:
                ax.scatter(b[te,0],b[te,1],c=color,marker='x',s=65)
        ax.set(title=f'{name} | {len(columns)} features\nPCA variance: {pca.explained_variance_ratio_.sum():.1%}',xlabel='PC1',ylabel='PC2')
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"Subject {data['subject']:02d}, pooled sessions, outer fold {fold['fold']}\n{boundary_label(boundary)} — circles=train, X=held out; scaler/PCA fitted on train")
    fig.tight_layout()
    stem=Path(root)/f"pca_sub-{data['subject']:02d}_{boundary}_k-{config['pca_k']}"
    fig.savefig(stem.with_suffix('.png'),dpi=160,bbox_inches='tight')
    save_json(stem.with_suffix('.json'),dict(subject=data['subject'],selection_boundary=boundary_label(boundary),
               selector_fit_rows=fit_rows.tolist(),pca_fit_rows=train.tolist(),test_rows=test.tolist(),
               k_plot_only=config['pca_k'],not_used_for_model_selection=True))
    plt.show()
    return fig

def plot_sweep(scores, root):
    sweep=scores[(scores.stage=='sweep')&(scores.scope=='pooled_subject')].copy() if not scores.empty else pd.DataFrame()
    if sweep.empty:
        print('K sweep is disabled or not yet completed. Enable CONFIG["run_sweep"].')
        return
    for feature_mode,frame in sweep.groupby('feature_mode'):
        subjects=sorted(frame.subject.unique())
        for metric in ('accuracy','balanced_accuracy'):
            fig,axes=plt.subplots(1,len(subjects)+1,figsize=(max(8,4*(len(subjects)+1)),4),squeeze=False)
            for boundary,g in frame.groupby('selection_boundary'):
                means=g.groupby('k')[metric].mean().sort_index()
                axes[0,0].plot(means.index,means.values,marker='o',label=boundary)
            axes[0,0].set_title('Equal-weight subject mean')
            for ax,subject in zip(axes[0,1:],subjects):
                for boundary,g in frame[frame.subject==subject].groupby('selection_boundary'):
                    g=g.sort_values('k')
                    ax.plot(g.k,g[metric],marker='o',label=boundary)
                ax.set_title(f'Subject {subject:02d}')
            for ax in axes[0]:
                ax.set(xlabel='K',ylabel=metric,ylim=(0,1))
                ax.grid(alpha=.25)
            axes[0,0].legend(fontsize=7)
            fig.suptitle(f'{feature_mode}: MRMR/SVM exploratory sweep; no outer-test selection of K')
            fig.tight_layout()
            fig.savefig(Path(root)/f'sweep_{feature_mode}_{metric}.png',dpi=130,bbox_inches='tight')
            plt.show()

def plot_stability(frequency, root, subject=None, k=10):
    if frequency.empty:
        print('No completed safe selections.')
        return
    subject=int(frequency.subject.min()) if subject is None else subject
    frame=frequency[(frequency.subject==subject)&(frequency.k==k)&(frequency.stage=='k10')]
    # One model per selector, since its fixed-K support is identical across classifier choices.
    for (mode,selector),g in frame.groupby(['feature_mode','selector']):
        first=g.result_id.iloc[0]
        g=g[g.result_id==first].nlargest(20,'selection_fraction').sort_values('selection_fraction')
        fig,ax=plt.subplots(figsize=(10,7))
        ax.barh(g.feature,g.selection_fraction)
        ax.set(xlim=(0,1),xlabel='Fraction of outer folds selected',title=f'Subject {subject:02d}, {mode}, {selector}, K={k} — NESTED_SAFE')
        fig.tight_layout()
        fig.savefig(Path(root)/f'stability_{subject}_{mode}_{selector}_k{k}.png',dpi=140,bbox_inches='tight')
        plt.show()

def plot_confusions(active, config, root, stage='k10'):
    for result in active.values():
        row=result['row']
        if row['stage']!=stage or row['scope']!='pooled_subject':
            continue
        cm=confusion_matrix(result['oof'].y_true,result['oof'].y_pred,labels=LABELS)
        fig,ax=plt.subplots(figsize=(5,4))
        ax.imshow(cm,cmap='Blues')
        for i in LABELS:
            for j in LABELS:
                ax.text(j,i,str(cm[i,j]),ha='center',va='center')
        ax.set(xticks=LABELS,yticks=LABELS,xticklabels=[config['class_names'][i] for i in LABELS],
               yticklabels=[config['class_names'][i] for i in LABELS],xlabel='Predicted',ylabel='True')
        ax.set_title(f"S{row['subject']:02d} {row['feature_mode']} {row['selector']} K={row['k']} {row['classifier']}\n{row['selection_boundary']}",fontsize=8)
        fig.tight_layout()
        fig.savefig(Path(root)/f"confusion_{row['result_id']}.png",dpi=130)
        plt.show()

def export_review_zip(active, config, root):
    # Include active runs only; do not bundle raw EEG, big feature matrices, or stale checkpoints.
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    path=Path(config['output_root'])/f'paper_replication_review_{stamp}.zip'
    root=Path(root)
    files=list(root.glob('*.csv'))+list(root.glob('*.json'))+list(root.glob('*.txt'))+list(root.glob('*.png'))
    for result in active.values():
        files.extend(p for p in result['directory'].iterdir() if p.suffix in {'.csv','.json'})
    with ZipFile(path,'w',ZIP_DEFLATED) as archive:
        for file in sorted(set(files)):
            archive.write(file,str(file.relative_to(root)))
    print(f'Review bundle: {path} ({path.stat().st_size/1024**2:.2f} MB)')
    return path


## 7. Validate configuration and start a fresh active result registry

Rerun this cell after changing the configuration. This notebook makes new folds automatically; no `SAVED_SPLITS_ROOT` or previous export is needed. Saved results are reused only for the exact matching run. The report includes only results evaluated/resumed in this execution, and records omitted optional models.


In [ ]:
from copy import deepcopy
RUN_CONFIG = deepcopy(CONFIG)
assert RUN_CONFIG["feature_mode"] in {"current", "paper_inspired"}
assert RUN_CONFIG["vmd_mode"] in {"off", "manual", "auto_approx"}
assert RUN_CONFIG["mrmr_relevance"] in {"mi", "f_stat"}
assert RUN_CONFIG["boundaries"] and set(RUN_CONFIG["boundaries"]).issubset({"PAPER_ORDER", "NESTED_SAFE"})
assert len(RUN_CONFIG["boundaries"]) == len(set(RUN_CONFIG["boundaries"]))
assert RUN_CONFIG["outer_folds"] == 10, "This requested protocol is 10-fold; do not silently change it."
assert RUN_CONFIG["inner_folds"] >= 2 and RUN_CONFIG["seed"] == 42
assert 1 <= RUN_CONFIG["pca_outer_fold"] <= RUN_CONFIG["outer_folds"]
assert RUN_CONFIG["pca_boundary"] in {"PAPER_ORDER", "NESTED_SAFE"}
assert len(RUN_CONFIG["lr5_members"]) == 5
assert RUN_CONFIG["blas_threads"] >= 1
assert RUN_CONFIG["k_values"] and all(isinstance(k,int) and k>0 for k in RUN_CONFIG["k_values"])
assert len(RUN_CONFIG["k_values"]) == len(set(RUN_CONFIG["k_values"]))
if RUN_CONFIG["full_sweep"]:
    RUN_CONFIG["k_values"] = [10] + list(range(20, 611))
RUN_CONFIG["data_root"] = discover_data_root(RUN_CONFIG["data_root"])
available = discover_subjects(RUN_CONFIG["data_root"])
if RUN_CONFIG["subjects"] is None:
    RUN_CONFIG["subjects"] = available
assert RUN_CONFIG["subjects"] and len(set(RUN_CONFIG["subjects"])) == len(RUN_CONFIG["subjects"])
assert set(RUN_CONFIG["subjects"]).issubset(available), f"Available subjects: {available}"
np.random.seed(RUN_CONFIG["seed"])
RUN_ROOT = Path(RUN_CONFIG["output_root"]) / ("experiment_" + digest(RUN_CONFIG)[:16])
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ACTIVE_RESULTS = {}
STAGE_STATUS = []
print("Dataset:", RUN_CONFIG["data_root"])
print("Subjects:", RUN_CONFIG["subjects"], "| Feature mode:", RUN_CONFIG["feature_mode"])
print("Output:", RUN_ROOT)
print("Versions:", environment_info())
print("MRMR:", RUN_CONFIG["mrmr_relevance"], "/ mean absolute Pearson; implementation assumption")
for mode in RUN_CONFIG["boundaries"]:
    print(boundary_label(mode))
save_json(RUN_ROOT / "source_inspection.json", SOURCE_INSPECTION)


## 8. Load and pool INNER SPEECH trials; inspect the dataset summary

Caching expensive features does not require any supervised full-data fit: both extractors are trial-local and label-free. The same cached matrix is reused across CV boundaries. EEG stays in volts until the explicit extractor unit conversion. Source hashes and actual channel names/rates/times are saved. `DATASETS[subject]` contains `F`, `y`, `metadata`, `schema`, `manifest` and `feature_key`.

This is the subject-level pool requested for replication; random folds mix that subject's sessions. It tests held-out trials from the available subject/session distribution, not chronological prediction, LOSO or leave-one-session-out.


In [ ]:
DATASETS = {}
for subject in RUN_CONFIG["subjects"]:
    DATASETS[subject] = prepare_subject_replication(subject, RUN_CONFIG)
SUMMARY = dataset_summary(DATASETS, RUN_ROOT)
display(SUMMARY)
for subject, data in DATASETS.items():
    print(f"Subject {subject:02d}: pooled EEG={data['manifest']['pooled_shape']}; F={data['F'].shape}; "
          f"classes={np.unique(data['y'], return_counts=True)}; VMD={data['manifest']['vmd_effective']}")
    plan = cv_plan(data, RUN_CONFIG)
    save_json(RUN_ROOT / f"sub-{subject:02d}_outer_plan.json", plan)


## 9. First and most important diagnostic: K=10

MRMR + SVM and ANOVA + optional LightGBM; ANOVA + SVM controls classifier choice. Every enabled boundary and subject is evaluated. Paper references: MRMR/SVM accuracy 66.53%, F1 67.09%; ANOVA/LightGBM accuracy 45.94%, F1 46.30%. **Reference accuracy is not BA**, and the paper's F1 averaging is not established as identical to macro F1 here.

Progress prints name the current boundary and fold. F-stat ranking is usually quicker than MI. A large number of feature candidates with few trials can give highly optimistic PAPER_ORDER results even when a classifier's own test fold is unseen.


In [ ]:
if RUN_CONFIG["run_k10"]:
    K10_RESULTS = run_stage(DATASETS, "k10", RUN_CONFIG, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
    display(K10_RESULTS[["subject", "feature_mode", "selector", "k", "classifier", "selection_boundary",
                         "accuracy", "balanced_accuracy", "macro_f1"]])
    display(aggregate_subjects(K10_RESULTS))
else:
    print("K10 disabled. Enable run_k10 for the main diagnostic.")


## 10. MRMR K sweep — exploratory, opt-in

Enable `run_sweep=True`. Defaults use the sparse specified grid; `full_sweep=True` is a dense 20–610 grid plus K10. Feature supports are computed once to maximum K for each exact fit set and reused as prefixes. Both selection boundaries use SVM. These plots cannot choose a final K with an unbiased performance claim: they use outer OOF results. Independent comparisons are descriptive.


In [ ]:
if RUN_CONFIG["run_sweep"]:
    SWEEP_RESULTS = run_stage(DATASETS, "sweep", RUN_CONFIG, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
    display(SWEEP_RESULTS)
    plot_sweep(SWEEP_RESULTS, RUN_ROOT)
else:
    print("Sweep disabled. Set run_sweep=True in CONFIG and rerun setup/downstream cells.")


## 11. Optional safe choice of K and SVM parameters

Enable `run_inner_k_choice=True`. In every outer training set, compare configured K values and SVM parameters with inner CV using training-only rankings/scalers. Refit the inner winner on all outer training and predict untouched outer test. This stage is NESTED_SAFE only; it does not pick the maximum from the plotted outer sweep.


In [ ]:
if RUN_CONFIG["run_inner_k_choice"]:
    INNER_SELECTED_RESULTS = run_stage(DATASETS, "inner_k_choice", RUN_CONFIG, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
    display(INNER_SELECTED_RESULTS)
else:
    print("Inner K selection disabled; other stages use their predeclared K values.")


## 12. Six classical models at fixed K=590

Enable `run_models590=True`. MRMR support is reused for each model at an identical fit boundary. Classifier parameters are configuration assumptions. LightGBM is skipped visibly if unavailable. Saved outputs include per-class recall, confusion matrices and subject means ± sample SD. Folds are never counted as independent subjects. Paper accuracy references are comparison context, not targets.


In [ ]:
if RUN_CONFIG["run_models590"]:
    MODEL590_RESULTS = run_stage(DATASETS, "models590", RUN_CONFIG, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
    display(MODEL590_RESULTS)
    display(aggregate_subjects(MODEL590_RESULTS))
else:
    print("Six-model K590 stage disabled; set run_models590=True.")


## 13. Five-LR bootstrap ensemble at fixed MRMR K=590

Enable `run_ensemble590=True`. Each bootstrap preserves training class counts and samples with replacement from outer TRAIN only. Each member has its own training scaler, predicts four aligned class probabilities and their mean gives the final prediction. The five valid configurations and OVR reduction are explicit implementation assumptions: **paper-inspired ensemble hyperparameters — exact configs not reported**. The paper reference is accuracy 81.13%, F1 81.12%.


In [ ]:
if RUN_CONFIG["run_ensemble590"]:
    ENSEMBLE_RESULTS = run_stage(DATASETS, "ensemble590", RUN_CONFIG, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
    display(ENSEMBLE_RESULTS)
    display(aggregate_subjects(ENSEMBLE_RESULTS))
else:
    print("Five-LR stage disabled; set run_ensemble590=True.")


## 14. Optional matched scope and representation comparisons

These provide evidence for CASE 3/4 instead of comparing incompatible historical experiments. `run_feature_mode_comparison` evaluates the other representation on the same subjects/trials/outer folds with both boundaries at K10. `run_matched_within_session` evaluates separate session models on those exact trials, also with 10 folds and the same selector/model settings. PAPER_ORDER uses full-session selection for those separate session models. No session is dropped and fold counts are not reduced when classes are too small: the notebook raises a clear error.

Pooling changes the number of training trials as well as training scope; a gain cannot isolate the role of stationarity. Feature-set improvement concerns this implemented approximation, not the unavailable exact 191 features.


In [ ]:
FEATURE_MODE_DATASETS = {RUN_CONFIG["feature_mode"]: DATASETS}
if RUN_CONFIG["run_feature_mode_comparison"]:
    other = "paper_inspired" if RUN_CONFIG["feature_mode"] == "current" else "current"
    compare_config = deepcopy(RUN_CONFIG)
    compare_config["feature_mode"] = other
    compare_data = {s: prepare_subject_replication(s, compare_config) for s in RUN_CONFIG["subjects"]}
    for s in DATASETS:
        assert DATASETS[s]["metadata"].trial_id.tolist() == compare_data[s]["metadata"].trial_id.tolist()
        assert np.array_equal(DATASETS[s]["y"], compare_data[s]["y"])
    FEATURE_MODE_DATASETS[other] = compare_data
    run_stage(compare_data, "k10", compare_config, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS)
if RUN_CONFIG["run_matched_within_session"]:
    for feature_mode, data_by_subject in FEATURE_MODE_DATASETS.items():
        compare_config = deepcopy(RUN_CONFIG)
        compare_config["feature_mode"] = feature_mode
        run_stage(data_by_subject, "k10", compare_config, RUN_ROOT, ACTIVE_RESULTS, STAGE_STATUS, scope="within_session")
if not RUN_CONFIG["run_feature_mode_comparison"] and not RUN_CONFIG["run_matched_within_session"]:
    print("Matched optional comparisons disabled; CASE 3/4 will be marked not assessed.")


## 15. PCA before selection, after ANOVA and after MRMR

Circles=train, X=held-out test, four class colors. StandardScaler and PCA fit **only the plotted outer train** in every panel; test is projected with those fitted transforms. Selected supports use the stated boundary. Default NESTED_SAFE avoids using test labels even for feature selection. PAPER_ORDER, if you explicitly request it, exposes labels at selection while PCA remains train-fitted.

`pca_k` controls the plot only; it does not set a training K. Explained variance is displayed; clean training clusters are not evidence of held-out classification performance. Keep the protocol frozen after looking at test crosses.


In [ ]:
if RUN_CONFIG["make_pca"]:
    plot_subject = RUN_CONFIG["pca_subject"] or next(iter(DATASETS))
    plot_pca(DATASETS[plot_subject], RUN_CONFIG, RUN_ROOT)
else:
    print("PCA disabled.")


## 16. Tables A–E, stability, session diagnostics and automatic interpretation

CSV scores use proportions (0.30 = 30%). Table A has equal-weight subject means and sample SD, Table B has one OOF metric row per subject/setting, C the sweep, D selected-feature frequency and a separate pairwise Jaccard table, and E paired boundary gaps. Per-session OOF scores are saved for every active model even though pooled models were trained across sessions. Skipped optional stages/models are recorded.

Default CASE thresholds are explicitly descriptive, not significance tests. CASE 1/2 compare matched K10 MRMR/SVM accuracies against the paper accuracy. CASE 3/4 need the optional matched runs; historical 25–30% BA is not substituted for a matched accuracy baseline. Stable selection may reflect artifacts/redundant features and **cannot prove physiology**. Save the protocol before choosing a follow-up confirmation experiment.


In [ ]:
TABLES = export_tables(ACTIVE_RESULTS, STAGE_STATUS, RUN_CONFIG, RUN_ROOT)
for name in ("A", "B", "C", "D", "E"):
    print(f"Table {name}")
    display(TABLES[name].head(30))
print("Per-session OOF diagnostics")
display(TABLES["sessions"].head(40))
if not TABLES["jaccard"].empty:
    display(TABLES["jaccard"].groupby(["subject", "feature_mode", "selector", "k", "classifier"])
            .jaccard.agg(["mean", "std"]))
plot_stability(TABLES["D"], RUN_ROOT, subject=RUN_CONFIG["pca_subject"], k=10)
if RUN_CONFIG["make_confusion_plots"]:
    plot_confusions(ACTIVE_RESULTS, RUN_CONFIG, RUN_ROOT, stage="k10")


## 17. Save a small review ZIP to share

This ZIP contains tables, OOF predictions, per-fold selected feature names/indices, fold metrics, confusion CSVs, training-boundary/ensemble bootstrap audit JSON, configuration/provenance, diagnostics and plots. It excludes raw EEG, large cached feature matrices and binary fold checkpoints. Only the active evaluations are bundled. Download it from Kaggle's output panel and send it for analysis.

Run stages independently if needed, then rerun Tables A–E and this cell. Reusing exact checkpoints is allowed; changing hyperparameters from outer plots and treating the score as confirmatory is not.


In [ ]:
REVIEW_ZIP = export_review_zip(ACTIVE_RESULTS, RUN_CONFIG, RUN_ROOT)
try:
    from IPython.display import FileLink
    display(FileLink(str(REVIEW_ZIP)))
except ImportError:
    print(REVIEW_ZIP)


## How to configure your next run

1. **Initial test:** `subjects=[1]`, leave `run_k10=True`, both boundaries, VMD off. Inspect dataset summary and K10 gap first.
2. **Cohort:** `subjects=None` to discover all subjects. Default summary weights subjects equally.
3. **Representation check:** `run_feature_mode_comparison=True`; this evaluates both feature modes at K10 on identical trials/folds.
4. **Scope check:** `run_matched_within_session=True`; do not infer pooling effects from unrelated old results.
5. **Larger models:** enable `run_models590` and `run_ensemble590`. Enable `run_sweep` separately if you want descriptive curves. `full_sweep` is much slower.
6. **Valid K choice:** enable `run_inner_k_choice`; never choose K from outer OOF/PCA plots.
7. **VMD experiment:** select manual channels or `auto_approx` before the run. Check the **effective** status, not just requested mode. An unavailable backend means VMD was skipped.

All settings are in CONFIG near the top. Rerun setup and subsequent cells after edits. The notebook is independent of old splits and prior feature exports. Remaining paper ambiguities: exact T1–T3 features/wavelet/fractal length, exact VMD thresholds/scope/parameters, MRMR estimator details, fold split implementation, most ML hyperparameters, exact five ensemble configurations and F1 averaging. No numerical match is promised.
